# 09 — Validation: Binding Affinity + ADMET Filtering

**Goal:** Score all candidates from notebooks 07-08 and filter for drug-likeness.

Pipeline:
1. **Boltz-2** — binding affinity prediction (built into structure prediction)
2. **DeltaForge** — thermodynamic scoring (r=0.83 vs experimental)
3. **RDKit** — Lipinski's Rule of 5, molecular properties
4. **OpenADMET** — ADMET prediction (absorption, distribution, metabolism, excretion, toxicity)
5. **AutoDock Vina** — classical docking cross-validation
6. **Literature comparison** — benchmark against known ERAP2 inhibitor DG013A

**Verification targets:**
- Binding affinity < 100nM to mutant ERAP2
- Pass Lipinski's Rule of 5
- No predicted toxicity flags
- Comparable or better affinity than DG013A

**Run on:** Google Colab (Boltz-2 on GPU) + local (RDKit/OpenADMET on CPU)

In [ ]:
!pip install -q rdkit-pypi meeko vina
# OpenADMET: pip install openadmet (check latest install instructions)

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Draw
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

def comprehensive_druglikeness(smiles: str) -> dict:
    """Full drug-likeness assessment."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"smiles": smiles, "valid": False}
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotatable = Descriptors.NumRotatableBonds(mol)
    
    # Lipinski's Rule of 5
    lipinski = (mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    
    # Veber's rules (oral bioavailability)
    veber = (tpsa <= 140 and rotatable <= 10)
    
    # PAINS filter (pan-assay interference compounds)
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog(params)
    pains_alert = catalog.HasMatch(mol)
    
    return {
        "smiles": smiles,
        "valid": True,
        "mw": round(mw, 1),
        "logp": round(logp, 2),
        "hbd": hbd,
        "hba": hba,
        "tpsa": round(tpsa, 1),
        "rotatable_bonds": rotatable,
        "lipinski_pass": lipinski,
        "veber_pass": veber,
        "pains_alert": pains_alert,
        "overall_pass": lipinski and veber and not pains_alert,
    }

In [ ]:
# Score DG013A as baseline reference
# DG013A SMILES: TODO — fetch from literature or PubChem
DG013A_SMILES = ""  # Set after lookup

# Bestatin (known aminopeptidase inhibitor, DrugBank DB00560)
BESTATIN_SMILES = "CC(O)C(=O)NC(CC1=CC=CC=C1)C(O)=O"

reference_drugs = [
    ("Bestatin", BESTATIN_SMILES),
]
if DG013A_SMILES:
    reference_drugs.append(("DG013A", DG013A_SMILES))

print("Reference drug-likeness scores:")
for name, smi in reference_drugs:
    result = comprehensive_druglikeness(smi)
    print(f"\n{name}:")
    for k, v in result.items():
        if k != "smiles":
            print(f"  {k}: {v}")

In [ ]:
# TODO: Load candidates from notebooks 07 and 08
# candidates_peptide = pd.read_csv("ligandforge_candidates.csv")  # from notebook 07
# candidates_smallmol = pd.read_csv("druggpt_candidates.csv")    # from notebook 08

# Score all candidates
# results = [comprehensive_druglikeness(smi) for smi in all_candidates]
# df = pd.DataFrame(results)
# print(f"Total candidates: {len(df)}")
# print(f"Valid SMILES: {df['valid'].sum()}")
# print(f"Pass all filters: {df['overall_pass'].sum()}")

print("Load candidates from previous notebooks to score.")

In [ ]:
# Final ranking: combine binding affinity + drug-likeness
# Ranking criteria (weighted):
# 1. Predicted binding affinity (Boltz-2 / DiffDock score) — 40%
# 2. Drug-likeness (Lipinski + Veber) — 20%
# 3. No PAINS alerts — 15%
# 4. ADMET predictions (oral bioavailability, no tox flags) — 25%

# Final output: top 10-20 candidates ranked by composite score
# Compare against DG013A benchmark
print("Final ranking will be computed after all scoring is complete.")